# Von Mises navigation policy

A compact, self-contained illustration of a **von Mises** steering policy. An agent at `x_start` wants to reach a target `g`; each step it samples a heading from a von Mises distribution

```
theta | x  ~  V( mu = heading-to-target,  kappa )  ,
step  =  step_len * ( cos theta, sin theta )
```

so that the *mean* heading is always directed at the goal, and `kappa` controls how much the actual step deviates from that ideal direction (kappa = 0 is a uniform, unbiased-but-random walk; larger kappa is tighter and more goal-directed).

The figure has two rows, one column per `kappa` in `{0, 0.5, 2, 5}`:

- **Top row** — the exact von Mises density `V(theta | mu, kappa)` rendered as a radial modulation around the policy radius, with the goal direction marked.
- **Bottom row** — 10 independent trajectories produced by the stepwise von Mises sampling, starting from the fixed agent origin (black dot) toward the target (triangle).

This is a *policy* demo — it characterises the stochastic heading distribution a controller would use — and is the visual companion to the *likelihood* use of the von Mises distribution in `beetles.ipynb` and `zebra_escape.ipynb`.


## Figure style

All figures share one house style (Arial text, white figures, hairline axes): the figure cell renders a low-resolution image inline (100 dpi) and saves the 300 dpi vector file it was given through the explicit `save_path=` argument, under `figures/von_mises_policy/` (the saved file is gitignored).


In [ ]:
%matplotlib inline
from importlib.metadata import version
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

import unwrap

# The figure code lives in a sibling script next to this notebook
# (von_mises_policy_plots.py); make it importable regardless of the
# kernel's working directory (the kernel may run from notebooks/ or from
# the repository root).
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(unwrap.__file__)), "notebooks"))

import von_mises_policy_plots as plots

{package: version(package) for package in ("unwrap", "numpy", "matplotlib")}


In [ ]:
# One figure style for every figure (house convention: 100 dpi inline,
# 300 dpi vector file saved under figures/von_mises_policy/).
plots.set_style()


## 1. The steering policy

The policy and its configuration, kept inline: `goal_heading` aims the mean von Mises direction at the target `g` from the current position, `vonmises_pdf` is the exact density `V(theta | mu, kappa)`, and the config fixes the origin, target, the concentration levels in `kappas`, the step size and the (per-column) sampling seeds, so every panel is comparable.


In [ ]:
# -----------------------------
# Von Mises steering policy
# -----------------------------
def goal_heading(x, y):        # mean heading always points at the goal g
    d = y - x
    return np.arctan2(d[1], d[0])

def vonmises_pdf(theta, mu, kappa):   # exact von Mises density V(theta | mu, kappa)
    return np.exp(kappa * np.cos(theta - mu)) / (2.0 * np.pi * np.i0(kappa))

# -----------------------------
# Config
# -----------------------------
x_start = np.array([0.0, 0.0], dtype=np.float32)
target  = np.array([0.0, 10.0], dtype=np.float32)

kappas  = np.array([0, 0.5, 2, 5], dtype=np.float32)
T_steps, step_len, num_traj, rng_seed = 300, 0.15, 10, 12345
R_policy, amp = 1.75, 1.0


## 2. Evaluate the densities and sample trajectories

The top-row radial profile evaluates the exact von Mises density on a fine angle grid for each `kappa` (rescaled to a small radial modulation around the policy radius). The bottom-row trajectories are the Monte-Carlo stepwise sampling of the same policy - the two rows are two views of the *same* distribution family, and their agreement is the point of the figure.


In [ ]:
# -----------------------------
# Top row: exact von Mises density as a radial modulation
# -----------------------------
theta_grid = np.linspace(0, 2 * np.pi, 1024, endpoint=False)
mu0 = goal_heading(x_start, target)
radials = []
for kappa in kappas:
    pdf = vonmises_pdf(theta_grid, mu=mu0, kappa=float(kappa)).astype(np.float32)
    radials.append(R_policy + amp * (pdf - pdf.min()) / (pdf.max() - pdf.min() + 1e-12))

# -----------------------------
# Bottom row: stepwise von Mises Monte-Carlo trajectories
# -----------------------------
trajectories = []
for i, kappa in enumerate(kappas):
    rng = np.random.default_rng(rng_seed + i)
    M = num_traj
    pos = np.empty((T_steps + 1, M, 2), dtype=np.float32)
    pos[0, :, :] = x_start[None, :]
    for t in range(T_steps):
        x_t = pos[t]
        mu_t = np.arctan2(target[None, 1] - x_t[:, 1], target[None, 0] - x_t[:, 0])
        th_t = rng.vonmises(mu=mu_t, kappa=float(kappa), size=M).astype(np.float32)
        pos[t + 1] = x_t + np.stack([np.cos(th_t), np.sin(th_t)], axis=-1) * step_len
    trajectories.append(pos)


## 3. Draw the figure

One call: the plotting script draws the two-row figure and saves it (300 dpi vector) under `figures/von_mises_policy/`; the cell renders it inline at 100 dpi.


In [ ]:
# The exact densities (top) and the Monte-Carlo trajectories (bottom),
# one column per kappa.
plots.plot_navigation(x_start, target, kappas, theta_grid, radials, trajectories,
                      R_policy=R_policy, amp=amp,
                      save_path="figures/von_mises_policy/vonmises_navigation.pdf")
plt.show()


## How to read the panels

- **kappa = 0**: the top-row curve is a perfect circle (the density is uniform in theta) and the bottom-row trajectories are a random walk — no net progress on average.
- **kappa = 0.5 / 2 / 5**: the radial curve progressively sharpens in the direction of the goal (a lobe on the goal side), and the trajectories progressively align toward the target. By `kappa = 5` the walk makes steady, low-deviation progress.
- The fixed `step_len`, `T_steps`, and origin keep the panels directly comparable, isolating the single effect of concentration `kappa`.

Note the two rows are drawn from the *same* distribution family: the top row is the analytic PDF; the bottom row is the Monte-Carlo realisation of that PDF integrated over time. Agreement between them is the point.
